# Results for model: mistralai/ministral-14b-instruct-2512

In [ ]:
import polars as pl
import xgboost as xgb
from lightgbm import early_stopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np

def compute_top_quantile(df: pl.DataFrame) -> pl.DataFrame:
    quantile = 0.85
    rolling_window = 90
    df = df.with_columns(
        pl.col("date_id").alias("date_a"),
        pl.col("date_id").alias("date_b"),
        pl.lit(quantile).alias("quantile"),
        pl.lit(quantile * 100).alias("rt_quantile_a")
    )

    for future in range(4):
        for responder in range(1, 6):
            colsuffix = f"_f{future}_r{responder}"
            vals = [f"feature_{i}" for i in range(252)]
            rank_cols = []
            for val in vals:
                rank_col = ((pl.col(val[0]) - pl.col("feature_0").over("date_a")) /
                            pl.col(val[0]).std().over("date_a")).rank("desc").over("date_a")
                rank_cols.append(rank_col.alias(f"rank_{val}"))

            df = df.with_columns(pl.struct(rank_cols).alias(f"rank_features"))

            ordered_df = df.sort(f"rank_feature_0", descending=True)
            top_quantile = quantile * 100
            top_quantile_b = top_quantile * (future + 1)

            df = df.join(
                ordered_df.filter(pl.col(f"rank_feature_0") <= top_quantile_b)
                              .select(pl.col("date_id"), pl.col("feature_0").suffix("_quantile_" + str(responder)))
                              .rename({"feature_0_quantile_" + str(responder): f"feature_0_quantile_a_responder_{responder}"}))
            )

        for base in range(1, 7):
            base_df = df.filter(pl.col("date_id") -  base  >= 0)
            if base_df.is_empty():
                break

            df = base_df.join(
                df.filter(pl.col("date_id") == base)
                  .select([f"responder_{i}" for i in range(6)])
                  .rename({f"responder_{i}": f"responder_{i}_h{base}"})
            )

    return df

def main():
    np.random.seed(42)
    df = pl.read_parquet('./jane_street_data/train.parquet')

    for percentage in [0.85, 0.95]:
        handle_quantile = 0 if "quantile" in df.columns else True
        additional_columns = []

        for future in [0, 1, 2, 3]:
            for responder in range(1, 7):
                responder_column = f"responder_{responder}"
                target_col = f"target_{future}"

                if handle_quantile:
                    quant_col = f"quantile_responder_{responder}"
                    mask = (df[quant_col] > percentage) | (df[quant_col] <= 0)
                    df = df.with_columns(pl.when(mask).then(0.0).otherwise(df[quant_col]).alias(quant_col))

                    additional_columns.append(quant_col)

                df = df.with_columns(
                    pl.from_numpy(plt.quantile(np.random.uniform(0, 1, len(df)), percentage * 0.5))
                    .alias(quant_col)
                )
                df = df.with_columns((df[quant_col] <= df[responder_column]).alias(target_col))
                target_col_dict = {}
                for future_col in range(future + 1):
                    f_prefix = f"f{future_col}"
                    target_col_dict[responder_column] = plКогда(f"{f_prefix}_{responder_column}")

    additional_columns = set(additional_columns)
    features = [col for col in df.columns if pl.Utf8ama-9.is_infty(col, "feature")]
    quantiles = [(multiply(
        (pl.col(col) - df[col].median()).abs()) for col in additional_columns)

    handle_quantiles(df.select([col for col in features]))

    features = [pl.select(pl.col * (1.0 / 3) if "quantile" not in col else 1.0 else pl.col(col)) for col in features]
    target_columns = [f"target_{i}" for i in range(4)]

    df = df.select(features + target_columns)

    df = df.with_columns